# 🚗 拍照估車價：二手車 AI 估價系統（Colab 完整版）

**拍一張車子照片 → AI 辨識品牌 → 輸入車況 → 預測二手價格**

## 使用方式
1. 上方選單「**執行階段**」→「**變更執行階段類型**」→ 選 **T4 GPU** → 儲存
2. **從上到下，一格一格按 ▶ 執行**（等上一格跑完再跑下一格）
3. 全部跑完大約 **45～70 分鐘**，期間不要關掉分頁

| 步驟 | 內容 | 時間 |
|---|---|---|
| 0 | 檢查 GPU、建立專案檔案、安裝套件 | 2 分鐘 |
| 1 | 數值：下載價格資料、訓練價格模型 | 5～10 分鐘 |
| 2 | 圖片：下載車子照片、訓練品牌辨識 | 30～45 分鐘 |
| 3 | 用自己拍的照片測試 | 隨時 |
| 4 | **在 Colab 裡直接打開網站** | 1 分鐘 |
| 5 | 下載模型／存到雲端硬碟 | 1 分鐘 |

> 需要準備：**Kaggle API Token**（Kaggle 網站 → 右上角頭像 → Settings → API → Create New Token）
>
> 💡 **第二次以後打開**：只要執行「步驟 0」+「5-3 從雲端硬碟載回模型」，就可以直接跳到步驟 3、4，不用重新訓練。

## 步驟 0：準備環境
### 0-1 檢查 GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("✅ 有 GPU：" + str(gpus) if gpus else "❌ 沒有 GPU！請到「執行階段 → 變更執行階段類型」選 T4 GPU，再重新執行這格")

### 0-2 建立專案資料夾

In [ ]:
%cd /content
!mkdir -p used-car-price/templates used-car-price/static used-car-price/data used-car-price/models
%cd /content/used-car-price

### 0-3 寫入程式檔案
下面每一格會產生一個程式檔，**全部執行即可，不用修改**。執行後會顯示 `Writing xxx.py` 或 `Overwriting xxx.py`。

> 設定檔叫 `car_config.py` 而不是 `config.py`：Colab 環境裡已經有一個叫 `config` 的模組，同名會載入錯的檔案。

In [ ]:
%%writefile car_config.py
"""專案設定。"""
import os

DATA_DIR = "data"
MODEL_DIR = "models"

# ===== 二手車價格資料（Kaggle：100,000 UK Used Car Data set）=====
PRICE_DATA_DIR = os.path.join(DATA_DIR, "used_cars")
PRICE_MODEL_DIR = os.path.join(MODEL_DIR, "price")

# 檔名 → 品牌（注意：原始資料把 Hyundai 拼成 hyundi）
PRICE_FILES = {
    "audi.csv": "Audi", "bmw.csv": "BMW", "ford.csv": "Ford", "hyundi.csv": "Hyundai",
    "merc.csv": "Mercedes-Benz", "skoda.csv": "Skoda", "toyota.csv": "Toyota",
    "vauxhall.csv": "Vauxhall", "vw.csv": "Volkswagen",
}

# ===== 車子照片（Stanford Cars，從 Hugging Face 下載）=====
HF_DATASET = "tanganke/stanford_cars"
IMAGE_DATA_DIR = os.path.join(DATA_DIR, "car_images")
IMAGE_MODEL_DIR = os.path.join(MODEL_DIR, "image")
IMG_SIZE = 224

# 圖片要辨識的品牌：兩份資料都有的 7 個品牌 + 「其他品牌」
IMAGE_BRANDS = ["Audi", "BMW", "Ford", "Hyundai", "Mercedes-Benz", "Toyota", "Volkswagen"]
OTHER = "Other"              # 資料夾用英文（Windows + TensorFlow 讀中文路徑可能出問題），畫面上顯示成中文


def display(name):
    """類別名稱 → 畫面上顯示的名稱。"""
    return "其他品牌" if name == OTHER else name
OTHER_MAX_PER_SPLIT = 1200   # 「其他品牌」照片很多，抽一部分就好，避免類別嚴重不平衡

# ===== 顯示用 =====
GBP_TO_TWD = 44.0            # 英鎊兌台幣（約略值，2026/11 約 44，可自行更新）
KM_PER_MILE = 1.609


In [ ]:
%%writefile plot_font.py
"""讓 matplotlib 圖表能顯示中文（Windows / Mac / Colab 都適用）。"""
import glob

import matplotlib.pyplot as plt
from matplotlib import font_manager

CANDIDATES = ["Microsoft JhengHei", "PingFang TC", "Noto Sans CJK TC", "Noto Sans CJK JP",
              "Noto Sans TC", "Heiti TC", "Arial Unicode MS"]


def setup_chinese_font():
    # Colab / Linux：裝了 fonts-noto-cjk 後，字型檔不一定被 matplotlib 自動登記，手動加入
    for path in glob.glob("/usr/share/fonts/**/*CJK*", recursive=True):
        try:
            font_manager.fontManager.addfont(path)
        except Exception:
            pass
    available = {f.name for f in font_manager.fontManager.ttflist}
    found = [c for c in CANDIDATES if c in available]
    if not found:
        print("⚠ 找不到中文字型，圖表中文會變方框。"
              "Colab 請先執行：!apt-get -qq install -y fonts-noto-cjk")
    plt.rcParams["font.sans-serif"] = found + ["DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False


In [ ]:
%%writefile price_utils.py
"""二手車價格資料：讀取、清理、特徵（訓練與網站共用）。"""
import os

import numpy as np
import pandas as pd

import car_config as config

CAT_COLS = ["brand", "model", "transmission", "fuelType"]
NUM_COLS = ["year", "mileage", "engineSize"]
FEATURES = CAT_COLS + NUM_COLS
# 不使用 tax（牌照稅）和 mpg（油耗）：一般使用者估價時不會知道這些數字


def load_price_data(data_dir=config.PRICE_DATA_DIR):
    """讀取各品牌 CSV，合併成一張表並清理。"""
    frames = []
    for fname, brand in config.PRICE_FILES.items():
        path = None
        for root, _, files in os.walk(data_dir):
            if fname in files:
                path = os.path.join(root, fname)
                break
        if path is None:
            print(f"  ⚠ 找不到 {fname}，略過 {brand}")
            continue
        df = pd.read_csv(path)
        df.columns = [c.strip() for c in df.columns]
        df = df.rename(columns={c: "tax" for c in df.columns if c.startswith("tax")})
        df["brand"] = brand
        frames.append(df)
    if not frames:
        raise SystemExit(f"在 {data_dir} 找不到任何價格資料，請先下載 Kaggle 資料集（見 README）")

    df = pd.concat(frames, ignore_index=True)
    n0 = len(df)
    for c in ["model", "transmission", "fuelType"]:
        df[c] = df[c].astype(str).str.strip()          # 原始資料的車型前面有空白，例如 " A3"
    for c in ["year", "price", "mileage", "engineSize"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 清理明顯錯誤的資料
    df = df.dropna(subset=["year", "price", "mileage", "engineSize"])
    df = df[(df.year >= 1995) & (df.year <= 2020)]        # 資料裡有 2060 年這種錯字
    df = df[(df.engineSize > 0) & (df.engineSize < 7)]     # 排氣量 0 是缺值
    df = df[(df.price >= 500) & (df.price <= 150000)]
    df = df[df.mileage < 300000]
    df = df.drop_duplicates()
    print(f"讀取價格資料：{n0} 筆 → 清理後 {len(df)} 筆，{df.brand.nunique()} 個品牌")
    return df.reset_index(drop=True)


def build_preprocessor():
    """類別欄位 One-Hot、數值欄位標準化。"""
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), CAT_COLS),
        ("num", StandardScaler(), NUM_COLS),
    ])


def make_input(brand, model, year, mileage_km, transmission, fuel, engine):
    """網站輸入 → 模型輸入（公里轉英里）。"""
    return pd.DataFrame([{
        "brand": brand, "model": model, "year": float(year),
        "mileage": float(mileage_km) / config.KM_PER_MILE,
        "transmission": transmission, "fuelType": fuel, "engineSize": float(engine),
    }])[FEATURES]


In [ ]:
%%writefile train_price.py
"""
數值部分：二手車價格預測（回歸）

比較 4 個模型（預測的是 log(價格)，因為價格分布右偏，取 log 後誤差比較平均）：
  0. 基準線：同品牌、同車型、同年份的「平均價格」（最簡單的估價方式）
  1. Scikit-Learn  Linear Regression
  2. Scikit-Learn  Random Forest
  3. Keras         神經網路（Dense）

用法：python train_price.py
輸出：models/price/ 底下的模型、metrics.json、圖表
"""
import json
import os

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split
import keras
from keras import layers

import car_config as config
from plot_font import setup_chinese_font
from price_utils import FEATURES, build_preprocessor, load_price_data

setup_chinese_font()
OUT = config.PRICE_MODEL_DIR
out = lambda n: os.path.join(OUT, n)


def evaluate(y_true, y_pred):
    return {"MAE_£": round(float(mean_absolute_error(y_true, y_pred)), 1),
            "MAPE_%": round(float(mean_absolute_percentage_error(y_true, y_pred) * 100), 2),
            "R2": round(float(r2_score(y_true, y_pred)), 4)}


def build_nn(n_in, y_mean, y_std):
    """
    最後一層 Rescaling 把輸出換算回 log(價格)：輸出 = 網路結果 × 標準差 + 平均。
    第一版沒有這層，網路要從 0 慢慢學到 9~10（log 價格的大小），第 1 輪 loss 高達 4.45、
    驗證 loss 忽高忽低，第 2 輪就被 EarlyStopping 停掉，誤差率 15.3% 比基準線還差。
    加上這層後，網路一開始就從「平均價」附近起步，只需要學「比平均貴多少、便宜多少」。
    （換算寫在模型裡面，所以 app.py 不用改）
    """
    model = keras.Sequential([
        layers.Input((n_in,)),
        layers.Dense(256, activation="relu"), layers.Dropout(0.1),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1),
        layers.Rescaling(scale=y_std, offset=y_mean),
    ])
    model.compile(optimizer=keras.optimizers.Adam(5e-4), loss="mse", metrics=["mae"])
    return model


def main():
    os.makedirs(OUT, exist_ok=True)
    df = load_price_data()
    X, y = df[FEATURES], df["price"].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, random_state=42)
    X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.15, random_state=42)
    print(f"訓練 {len(X_tr)}｜驗證 {len(X_va)}｜測試 {len(X_te)}")

    pre = build_preprocessor().fit(X_tr)
    P_tr, P_va, P_te = (pre.transform(a) for a in (X_tr, X_va, X_te))
    ly_tr, ly_va = np.log(y_tr), np.log(y_va)
    preds = {}

    # 0. 基準線：同品牌+車型+年份的平均價（查不到就退回同品牌+車型、再退回同品牌）
    tr = X_tr.assign(price=y_tr)
    g3 = tr.groupby(["brand", "model", "year"]).price.mean()
    g2 = tr.groupby(["brand", "model"]).price.mean()
    g1 = tr.groupby("brand").price.mean()
    preds["基準線(查平均價)"] = np.array([
        g3.get((b, m, yr), g2.get((b, m), g1.get(b, tr.price.mean())))
        for b, m, yr in zip(X_te.brand, X_te.model, X_te.year)])

    print("訓練 Linear Regression ...")
    lr = Ridge(alpha=1.0).fit(P_tr, ly_tr)          # 加一點正則化，避免罕見車型係數爆掉
    preds["LinearRegression"] = np.exp(lr.predict(P_te))

    print("訓練 Random Forest ...（只用 CPU，約 3~10 分鐘，請耐心等候）")
    rf = RandomForestRegressor(n_estimators=100, max_depth=25, min_samples_leaf=3, n_jobs=-1,
                               random_state=42).fit(P_tr, ly_tr)
    preds["RandomForest"] = np.exp(rf.predict(P_te))

    print("訓練 Keras 神經網路 ...")
    dense = lambda a: a.toarray().astype("float32") if hasattr(a, "toarray") else a.astype("float32")
    nn = build_nn(P_tr.shape[1], float(ly_tr.mean()), float(ly_tr.std()))
    hist = nn.fit(dense(P_tr), ly_tr, validation_data=(dense(P_va), ly_va), epochs=100, batch_size=256,
                  verbose=2, callbacks=[
                      keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
                      # 驗證 loss 停滯時自動把學習率減半，讓後面學得更細
                      keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-5, verbose=1)])
    preds["Keras神經網路"] = np.exp(nn.predict(dense(P_te), verbose=0).ravel())

    metrics = {k: evaluate(y_te, v) for k, v in preds.items()}
    print("\n========== 測試集結果 ==========")
    for k, v in metrics.items():
        print(f"  {k:<16} 平均誤差 £{v['MAE_£']:>7,.0f}   誤差率 {v['MAPE_%']:5.1f}%   R² {v['R2']:.3f}")

    # 各品牌的誤差（用最好的模型）
    best = min((k for k in metrics if k != "基準線(查平均價)"), key=lambda k: metrics[k]["MAE_£"])
    by_brand = (pd.DataFrame({"brand": X_te.brand.values, "err": np.abs(preds[best] - y_te) / y_te * 100})
                .groupby("brand").err.mean().round(1).to_dict())
    print(f"\n  最好的模型：{best}；各品牌誤差率(%)：{by_brand}")

    joblib.dump(pre, out("preprocessor.joblib"))
    joblib.dump(lr, out("linear_regression.joblib"))
    joblib.dump(rf, out("random_forest.joblib"), compress=3)
    nn.save(out("nn.keras"))

    # 網站下拉選單需要的選項
    options = {b: {"models": sorted(g.model.unique().tolist(), key=lambda m: -(g.model == m).sum()),
                   "fuel": sorted(g.fuelType.unique().tolist()),
                   "transmission": sorted(g.transmission.unique().tolist()),
                   "engine_median": float(g.engineSize.median())}
               for b, g in df.groupby("brand")}
    with open(out("metrics.json"), "w", encoding="utf-8") as f:
        json.dump({"metrics": metrics, "best": best, "by_brand_mape": by_brand,
                   "year_range": [int(df.year.min()), int(df.year.max())], "options": options},
                  f, ensure_ascii=False, indent=2)

    plots(metrics, preds, y_te, X_te, hist, rf, pre)
    print(f"\n✅ 完成！模型與圖表在 {OUT}/")


def plots(metrics, preds, y_te, X_te, hist, rf, pre):
    names = list(metrics)
    # 圖 1：模型比較
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, [metrics[n]["MAPE_%"] for n in names],
                   color=["#999", "#4C72B0", "#55A868", "#C44E52"])
    plt.bar_label(bars, fmt="%.1f%%"); plt.ylabel("平均誤差率 %（越低越好）")
    plt.title("二手車價格預測：模型比較"); plt.tight_layout()
    plt.savefig(out("1_model_compare.png"), dpi=150); plt.close()

    # 圖 2：預測 vs 實際
    best = min((k for k in metrics if "基準" not in k), key=lambda k: metrics[k]["MAE_£"])
    plt.figure(figsize=(5.5, 5.5))
    plt.scatter(y_te, preds[best], s=3, alpha=0.3)
    lim = [0, np.percentile(y_te, 99.5)]
    plt.plot(lim, lim, "r--", label="完美預測"); plt.xlim(lim); plt.ylim(lim)
    plt.xlabel("實際價格 (£)"); plt.ylabel("預測價格 (£)"); plt.legend()
    plt.title(f"預測 vs 實際（{best}）"); plt.tight_layout()
    plt.savefig(out("2_pred_vs_actual.png"), dpi=150); plt.close()

    # 圖 3：特徵重要性（One-Hot 的欄位加總回原本的特徵）
    names_oh = pre.get_feature_names_out()
    imp = pd.Series(rf.feature_importances_, index=names_oh)
    groups = imp.groupby(lambda n: n.split("__")[1].split("_")[0] if n.startswith("cat__")
                         else n.split("__")[1]).sum().sort_values()
    label = {"brand": "品牌", "model": "車型", "year": "年份", "mileage": "里程",
             "engineSize": "排氣量", "transmission": "變速箱", "fuelType": "燃料"}
    plt.figure(figsize=(6, 4))
    plt.barh([label.get(i, i) for i in groups.index], groups.values, color="#55A868")
    plt.title("哪些因素最影響二手車價格？（Random Forest）"); plt.tight_layout()
    plt.savefig(out("3_feature_importance.png"), dpi=150); plt.close()

    # 圖 4：神經網路訓練曲線
    plt.figure(figsize=(6, 4))
    plt.plot(hist.history["loss"], label="訓練 loss"); plt.plot(hist.history["val_loss"], label="驗證 loss")
    plt.xlabel("epoch"); plt.legend(); plt.title("Keras 神經網路訓練過程"); plt.tight_layout()
    plt.savefig(out("4_nn_training.png"), dpi=150); plt.close()


if __name__ == "__main__":
    keras.utils.set_random_seed(42)
    main()


In [ ]:
%%writefile prepare_car_images.py
"""
步驟：下載 Stanford Cars（Hugging Face），整理成「一個品牌一個資料夾」

    data/car_images/train/Audi/xxx.jpg
    data/car_images/train/BMW/...
    data/car_images/train/Other/...     ← 其他品牌（資料夾用英文名稱）
    data/car_images/test/...        ← 官方切好的測試集，訓練時完全不會用到

原始資料有 196 類（品牌+車型+年份），這裡只取「品牌」：
7 個價格資料也有的品牌 + 其他品牌（Honda、Nissan…等，用來讓系統知道「這不是能估價的品牌」）

用法：python prepare_car_images.py
"""
import os
import random
import shutil

import car_config as config

# 196 類名稱的第一個字通常是品牌，但有些品牌是兩個字
MULTI_WORD = ["AM General", "Aston Martin", "Land Rover", "Rolls-Royce"]


def brand_of(class_name):
    for b in MULTI_WORD:
        if class_name.startswith(b):
            return b
    return class_name.split()[0]


def target_of(class_name):
    b = brand_of(class_name)
    return b if b in config.IMAGE_BRANDS else config.OTHER


def export_split(ds_split, names, split, rng):
    out_root = os.path.join(config.IMAGE_DATA_DIR, split)
    rows = list(range(len(ds_split)))
    labels = list(ds_split["label"])
    targets = [target_of(names[l]) for l in labels]

    # 「其他品牌」只抽一部分
    other_idx = [i for i in rows if targets[i] == config.OTHER]
    keep_other = set(rng.sample(other_idx, min(len(other_idx), config.OTHER_MAX_PER_SPLIT)))
    keep = [i for i in rows if targets[i] != config.OTHER or i in keep_other]

    counts = {}
    for n, i in enumerate(keep, 1):
        item = ds_split[i]
        t = targets[i]
        d = os.path.join(out_root, t)
        os.makedirs(d, exist_ok=True)
        model_name = names[labels[i]].replace(" ", "_").replace("/", "-")
        item["image"].convert("RGB").save(os.path.join(d, f"{i:05d}_{model_name}.jpg"), quality=92)
        counts[t] = counts.get(t, 0) + 1
        if n % 1000 == 0:
            print(f"  {split}: {n}/{len(keep)}")
    return counts


def main(ds=None):
    if ds is None:
        from datasets import load_dataset, load_dataset_builder
        print(f"從 Hugging Face 下載 {config.HF_DATASET}（只下載 train / test，第一次需要幾分鐘）...")
        try:
            # 這份資料集另外附了加雜訊的版本（contrast、gaussian_noise…），只抓需要的兩份
            ds = load_dataset(config.HF_DATASET,
                              data_files={"train": "data/train-*", "test": "data/test-*"})
        except Exception as e:
            print(f"  只下載部分檔案失敗（{e}），改成下載完整資料集")
            ds = load_dataset(config.HF_DATASET)
    names = getattr(ds["train"].features["label"], "names", None)
    if not names:
        from datasets import load_dataset_builder
        names = load_dataset_builder(config.HF_DATASET).info.features["label"].names
    if os.path.exists(config.IMAGE_DATA_DIR):
        shutil.rmtree(config.IMAGE_DATA_DIR)
    rng = random.Random(42)
    for split in ["train", "test"]:
        counts = export_split(ds[split], names, split, rng)
        print(f"\n{split} 各品牌張數：")
        for b in config.IMAGE_BRANDS + [config.OTHER]:
            print(f"  {config.display(b):<15}{counts.get(b, 0):>6}")
    print(f"\n✅ 照片已整理到 {config.IMAGE_DATA_DIR}/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_image.py
"""
圖片部分：從照片辨識汽車品牌（7 個品牌 + 其他品牌）

比較兩種方法：
  1. Scikit-Learn：MobileNetV2 抽特徵（不訓練）+ Logistic Regression 分類
  2. Keras CNN  ：MobileNetV2 遷移學習 + 資料增強（DataGen），兩階段訓練

測試集：Stanford Cars 官方切好的測試集（訓練時完全不會用到）

用法：python train_image.py
      python train_image.py --quick     （快速檢查程式能不能跑）
"""
import argparse
import json
import os

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import keras
from keras import layers

import car_config as config
from plot_font import setup_chinese_font

setup_chinese_font()
S = config.IMG_SIZE
OUT = config.IMAGE_MODEL_DIR
out = lambda n: os.path.join(OUT, n)


def load_datasets(batch):
    train_dir = os.path.join(config.IMAGE_DATA_DIR, "train")
    test_dir = os.path.join(config.IMAGE_DATA_DIR, "test")
    if not os.path.isdir(train_dir):
        raise SystemExit("找不到照片，請先執行 python prepare_car_images.py")
    kw = dict(image_size=(S, S), batch_size=batch, label_mode="int")
    tr, va = keras.utils.image_dataset_from_directory(
        train_dir, validation_split=0.15, subset="both", seed=42, **kw)
    class_names = tr.class_names
    te = keras.utils.image_dataset_from_directory(test_dir, shuffle=False, class_names=class_names, **kw)
    return tr, va, te, class_names


def compact_cache(ds):
    """照片存成 uint8 快取在記憶體（只佔 float32 的 1/4），用的時候再轉回 float32。
    第一版直接 cache() float32，訓練+測試快取約 6GB，Colab 記憶體容易爆掉。"""
    return (ds.map(lambda x, y: (tf.cast(tf.clip_by_value(x, 0, 255), tf.uint8), y)).cache()
              .map(lambda x, y: (tf.cast(x, tf.float32), y)).prefetch(tf.data.AUTOTUNE))


def labels_of(ds):
    return np.concatenate([y.numpy() for _, y in ds])


def build_cnn(n_classes, pretrained=True):
    # 資料增強（= 課堂的 ImageDataGenerator）
    # 車子的品牌和顏色、亮度無關 → 這裡可以放心調亮度、對比（和空污主題剛好相反！）
    augment = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2),
    ], name="data_augmentation")
    base = keras.applications.MobileNetV2(input_shape=(S, S, 3), include_top=False,
                                          weights="imagenet" if pretrained else None)
    base.trainable = False
    inputs = keras.Input((S, S, 3))
    x = augment(inputs)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs), base


def extract_features(base, ds):
    """
    Scikit-Learn 用：MobileNetV2 當作「特徵抽取器」，每張照片變成 1280 維向量。
    ⚠ 特徵和標籤一定要在「同一次讀取」中一起取出！
      訓練資料每次讀取都會重新打亂順序，第一版分兩次讀，特徵和答案對不起來，準確率只有 16%（≈亂猜）。
    """
    inp = keras.Input((S, S, 3))
    x = layers.Rescaling(1 / 127.5, offset=-1)(inp)
    x = layers.GlobalAveragePooling2D()(base(x, training=False))
    fe = keras.Model(inp, x)
    F, Y = [], []
    for b, y in ds:
        F.append(fe.predict_on_batch(b)); Y.append(y.numpy())
    return np.concatenate(F), np.concatenate(Y)


def main(args):
    os.makedirs(OUT, exist_ok=True)
    tr, va, te, class_names = load_datasets(args.batch)
    if args.quick:
        tr, va = tr.take(5), va.take(2)
    n = len(class_names)
    labels = [config.display(c) for c in class_names]      # 畫面/圖表顯示用（Other → 其他品牌）
    print("類別：", labels)

    y_tr = labels_of(tr)
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(n), y=y_tr)))
    te_paths = list(te.file_paths)          # 記下測試照片路徑，畫範例圖時只讀需要的幾張
    tr, va, te = compact_cache(tr), compact_cache(va), compact_cache(te)
    y_te = labels_of(te)

    model, base = build_cnn(n, pretrained=not args.no_pretrained)

    # ===== 1. Scikit-Learn：CNN 特徵 + Logistic Regression =====
    print("\n[1/2] Scikit-Learn：用 MobileNetV2 抽特徵 + Logistic Regression ...")
    F_tr, yf_tr = extract_features(base, tr)
    F_te, _ = extract_features(base, te)
    clf = LogisticRegression(max_iter=2000, C=0.5, class_weight="balanced").fit(F_tr, yf_tr)
    pred_lr = clf.predict(F_te)
    joblib.dump(clf, out("logreg_on_cnn_features.joblib"))

    # ===== 2. Keras：遷移學習 =====
    print("\n[2/2] Keras：MobileNetV2 遷移學習")
    ckpt = out("car_brand_cnn.keras")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h1 = model.fit(tr, validation_data=va, epochs=args.epochs, class_weight=cw, verbose=2,
                   callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
    stage1_val = max(h1.history["val_accuracy"])
    model.save(ckpt)

    # 第二階段微調：只有在驗證準確率「真的變好」時才採用（上一個主題學到的教訓）
    base.trainable = True
    for layer in base.layers[:-40]:
        layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(1e-5),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h2 = model.fit(tr, validation_data=va, epochs=args.finetune_epochs, class_weight=cw, verbose=2,
                   callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])
    stage2_val = max(h2.history["val_accuracy"]) if h2.history else 0
    if stage2_val > stage1_val:
        model.save(ckpt)
        print(f"  ✔ 微調有效（驗證 {stage1_val:.3f} → {stage2_val:.3f}），採用微調後的模型")
    else:
        model = keras.models.load_model(ckpt)
        print(f"  ✘ 微調沒有變好（{stage1_val:.3f} → {stage2_val:.3f}），保留第一階段的模型")

    prob = model.predict(te, verbose=0)
    pred_cnn = prob.argmax(1)

    # ===== 評估 =====
    results = {}
    print("\n========== 測試集結果（Stanford Cars 官方測試集）==========")
    for name, pred in [("LogReg(CNN特徵)", pred_lr), ("CNN遷移學習", pred_cnn)]:
        results[name] = {
            "accuracy": round(float(accuracy_score(y_te, pred)), 4),
            "macro_f1": round(float(f1_score(y_te, pred, average="macro")), 4)}
        print(f"\n{name}：{results[name]}")
        print(classification_report(y_te, pred, labels=range(n), target_names=labels, zero_division=0))
    top2 = float(np.mean([y in p.argsort()[-2:] for y, p in zip(y_te, prob)]))
    results["CNN遷移學習"]["top2_accuracy"] = round(top2, 4)
    print(f"CNN 前兩名包含正確答案的比例（Top-2）：{top2:.1%}")

    with open(out("metrics.json"), "w", encoding="utf-8") as f:
        json.dump({"class_names": class_names, "results": results, "n_test": int(len(y_te))},
                  f, ensure_ascii=False, indent=2)
    plot_confusion(y_te, pred_cnn, labels)
    plot_history(h1, h2)
    plot_examples(te_paths, prob, y_te, labels)
    print(f"\n✅ 完成！模型與圖表在 {OUT}/")


def plot_confusion(y, pred, names):
    cm = confusion_matrix(y, pred, labels=range(len(names)))
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xticks(range(len(names)), names, rotation=35, ha="right")
    ax.set_yticks(range(len(names)), names)
    ax.set_xlabel("模型預測"); ax.set_ylabel("實際品牌"); ax.set_title("品牌辨識混淆矩陣（CNN）")
    fig.tight_layout(); fig.savefig(out("1_confusion_matrix.png"), dpi=150); plt.close(fig)


def plot_history(h1, h2):
    acc = h1.history["accuracy"] + h2.history.get("accuracy", [])
    val = h1.history["val_accuracy"] + h2.history.get("val_accuracy", [])
    plt.figure(figsize=(7, 4))
    plt.plot(acc, label="訓練準確率"); plt.plot(val, label="驗證準確率")
    plt.axvline(len(h1.history["accuracy"]) - 0.5, color="gray", ls="--", label="開始微調")
    plt.xlabel("epoch"); plt.legend(); plt.title("CNN 訓練過程")
    plt.tight_layout(); plt.savefig(out("2_cnn_training.png"), dpi=150); plt.close()


def plot_examples(paths, prob, y, names, k=10):
    """只讀抽到的 k 張照片（第一版把整個測試集讀進記憶體，Colab 會爆掉）。"""
    idx = np.random.default_rng(0).choice(len(y), size=min(k, len(y)), replace=False)
    fig, axes = plt.subplots(2, (len(idx) + 1) // 2, figsize=(15, 6.5))
    for ax, i in zip(axes.ravel(), idx):
        ax.imshow(plt.imread(paths[i])); ax.axis("off")
        p = prob[i].argmax()
        ax.set_title(f"實際：{names[y[i]]}\n預測：{names[p]}（{prob[i][p]:.0%}）", fontsize=9,
                     color="green" if p == y[i] else "red")
    fig.tight_layout(); fig.savefig(out("3_examples.png"), dpi=120); plt.close(fig)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--finetune-epochs", type=int, default=25)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--no-pretrained", action="store_true", help="不下載 ImageNet 權重（只用於離線測試）")
    a = ap.parse_args()
    if a.quick:
        a.epochs, a.finetune_epochs = 1, 1
    keras.utils.set_random_seed(42)
    main(a)


### 2-5（選做）接著目前的模型繼續訓練
如果 2-3 最後幾輪 `val_accuracy` 還在上升，可以接著練，不用從頭來。
- 會先**備份**目前的模型；**驗證準確率有進步才採用**，沒進步就保留原本的
- 想多學一點：加 `--unfreeze 80`（解凍更多層，但比較容易過擬合）
- 練壞了想還原：`continue_train.py --restore`

In [ ]:
!python continue_train.py --epochs 20

In [ ]:
%%writefile continue_train.py
"""
接著目前的品牌辨識模型繼續訓練（不用從頭來）

    python continue_train.py                     再練 20 輪（解凍最後 40 層，和原本一樣）
    python continue_train.py --epochs 30         再練 30 輪
    python continue_train.py --unfreeze 80       多解凍一些層（模型能學更多，但也更容易過擬合）
    python continue_train.py --restore           練壞了？還原成繼續訓練前的模型

保護機制：
  1. 開始前先備份目前的模型（car_brand_cnn_backup.keras）
  2. 用「驗證集」判斷有沒有變好：變好才採用，沒變好就保留原本的模型
     （不拿測試集挑模型，否則測試分數就不公平了）
  3. 驗證集和 train_image.py 用同一個 seed 切，所以訓練時沒看過的照片還是沒看過
"""
import argparse
import json
import os
import shutil

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
import keras

import car_config as config
import train_image as T

MODEL = T.out("car_brand_cnn.keras")
BACKUP = T.out("car_brand_cnn_backup.keras")
METRICS = T.out("metrics.json")


def set_trainable(model, n_unfreeze):
    """解凍 MobileNetV2 最後 n 層（BatchNorm 保持凍結，小資料微調比較穩定）。"""
    base = next(l for l in model.layers if isinstance(l, keras.Model) and "mobilenet" in l.name.lower())
    base.trainable = True
    for layer in base.layers[:-n_unfreeze]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False


def main(args):
    if args.restore:
        if not os.path.exists(BACKUP):
            raise SystemExit("找不到備份檔，沒辦法還原")
        shutil.copy(BACKUP, MODEL)
        raise SystemExit("✅ 已還原成繼續訓練前的模型（測試集分數請重跑 continue_train.py --epochs 0 更新）")
    if not os.path.exists(MODEL):
        raise SystemExit("找不到模型，請先執行 python train_image.py")

    tr, va, te, class_names = T.load_datasets(args.batch)
    labels = [config.display(c) for c in class_names]
    n = len(class_names)
    y_tr = T.labels_of(tr)
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(n), y=y_tr)))
    te_paths = list(te.file_paths)
    tr, va, te = T.compact_cache(tr), T.compact_cache(va), T.compact_cache(te)
    y_te = T.labels_of(te)

    model = keras.models.load_model(MODEL)
    shutil.copy(MODEL, BACKUP)
    print(f"已備份目前的模型 → {BACKUP}")

    model.compile(optimizer=keras.optimizers.Adam(args.lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    _, val_before = model.evaluate(va, verbose=0)
    print(f"目前模型的驗證準確率：{val_before:.3f}")

    hist = None
    if args.epochs > 0:
        set_trainable(model, args.unfreeze)
        model.compile(optimizer=keras.optimizers.Adam(args.lr),   # 改了可訓練層之後要重新 compile
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        print(f"\n繼續訓練 {args.epochs} 輪（解凍最後 {args.unfreeze} 層，學習率 {args.lr}）")
        hist = model.fit(tr, validation_data=va, epochs=args.epochs, class_weight=cw, verbose=2, callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, verbose=1)])
        _, val_after = model.evaluate(va, verbose=0)
        if val_after > val_before:
            model.save(MODEL)
            print(f"\n✔ 驗證準確率 {val_before:.3f} → {val_after:.3f}，有進步，採用新模型")
        else:
            model = keras.models.load_model(BACKUP)
            print(f"\n✘ 驗證準確率 {val_before:.3f} → {val_after:.3f}，沒有進步，保留原本的模型")

    # ===== 用測試集評估最後採用的模型 =====
    prob = model.predict(te, verbose=0)
    pred = prob.argmax(1)
    res = {"accuracy": round(float(accuracy_score(y_te, pred)), 4),
           "macro_f1": round(float(f1_score(y_te, pred, average="macro")), 4),
           "top2_accuracy": round(float(np.mean([y in p.argsort()[-2:] for y, p in zip(y_te, prob)])), 4)}
    print("\n========== 測試集結果（Stanford Cars 官方測試集）==========")
    print(f"CNN遷移學習：{res}")
    print(classification_report(y_te, pred, labels=range(n), target_names=labels, zero_division=0))

    with open(METRICS, encoding="utf-8") as f:
        meta = json.load(f)
    meta["results"]["CNN遷移學習"] = res
    with open(METRICS, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    T.plot_confusion(y_te, pred, labels)
    T.plot_examples(te_paths, prob, y_te, labels)
    if hist is not None:
        plt.figure(figsize=(7, 4))
        plt.plot(hist.history["accuracy"], label="訓練準確率")
        plt.plot(hist.history["val_accuracy"], label="驗證準確率")
        plt.axhline(val_before, color="gray", ls="--", label="繼續訓練前的驗證準確率")
        plt.xlabel("epoch（繼續訓練）"); plt.legend(); plt.title("繼續訓練過程")
        plt.tight_layout(); plt.savefig(T.out("4_continue_training.png"), dpi=150); plt.close()
    print(f"\n✅ 完成！metrics.json、混淆矩陣、範例圖都已更新（{config.IMAGE_MODEL_DIR}/）")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--unfreeze", type=int, default=40, help="解凍 MobileNetV2 最後幾層")
    ap.add_argument("--lr", type=float, default=1e-5)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--restore", action="store_true", help="還原成繼續訓練前的模型")
    a = ap.parse_args()
    keras.utils.set_random_seed(42)
    main(a)

In [ ]:
%%writefile app.py
"""
Flask 網站：拍照辨識品牌 → 輸入車況 → 估價
用法：python app.py  → 打開 http://127.0.0.1:5000
"""
import io
import json
import os
from functools import lru_cache

import joblib
import keras
import numpy as np
import tensorflow as tf
from flask import Flask, jsonify, render_template, request
from PIL import Image, ImageOps

import car_config as config
from price_utils import make_input

BASE = os.path.dirname(os.path.abspath(__file__))
os.chdir(BASE)          # 不管從哪裡啟動，都用專案資料夾當工作目錄（模型路徑才找得到）

app = Flask(__name__, template_folder=os.path.join(BASE, "templates"),
            static_folder=os.path.join(BASE, "static"))


@lru_cache(maxsize=1)
def price_models():
    d = config.PRICE_MODEL_DIR
    if not os.path.exists(os.path.join(d, "nn.keras")):
        return None
    with open(os.path.join(d, "metrics.json"), encoding="utf-8") as f:
        meta = json.load(f)
    return {"meta": meta,
            "pre": joblib.load(os.path.join(d, "preprocessor.joblib")),
            "LinearRegression": joblib.load(os.path.join(d, "linear_regression.joblib")),
            "RandomForest": joblib.load(os.path.join(d, "random_forest.joblib")),
            "Keras神經網路": keras.models.load_model(os.path.join(d, "nn.keras"))}


@lru_cache(maxsize=1)
def image_model():
    d = config.IMAGE_MODEL_DIR
    if not os.path.exists(os.path.join(d, "car_brand_cnn.keras")):
        return None
    with open(os.path.join(d, "metrics.json"), encoding="utf-8") as f:
        meta = json.load(f)
    return {"meta": meta, "cnn": keras.models.load_model(os.path.join(d, "car_brand_cnn.keras"))}


def predict_price(p, X):
    P = p["pre"].transform(X)
    dense = P.toarray().astype("float32") if hasattr(P, "toarray") else P.astype("float32")
    return {
        "LinearRegression": float(np.exp(p["LinearRegression"].predict(P)[0])),
        "RandomForest": float(np.exp(p["RandomForest"].predict(P)[0])),
        "Keras神經網路": float(np.exp(p["Keras神經網路"].predict(dense, verbose=0).ravel()[0])),
    }


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/api/options")
def options():
    p, im = price_models(), image_model()
    return jsonify({
        "price_ready": p is not None, "image_ready": im is not None,
        "options": p["meta"]["options"] if p else {},
        "year_range": p["meta"]["year_range"] if p else [2005, 2020],
        "price_metrics": p["meta"]["metrics"] if p else {},
        "best_price_model": p["meta"]["best"] if p else None,
        "image_metrics": im["meta"]["results"] if im else {},
        "gbp_to_twd": config.GBP_TO_TWD,
    })


@app.route("/api/recognize", methods=["POST"])
def recognize():
    im = image_model()
    if im is None:
        return jsonify({"error": "還沒有圖片模型，請先執行 python train_image.py"}), 400
    f = request.files.get("photo")
    if not f:
        return jsonify({"error": "請選擇一張照片"}), 400
    try:
        img = ImageOps.exif_transpose(Image.open(io.BytesIO(f.read()))).convert("RGB")
    except Exception:
        return jsonify({"error": "無法讀取這個檔案，請上傳 JPG 或 PNG 圖片"}), 400

    # 和訓練時相同：整張縮成 224x224（image_dataset_from_directory 的預設做法）
    x = tf.image.resize(np.asarray(img, dtype="float32"), (config.IMG_SIZE, config.IMG_SIZE))[None]
    prob = im["cnn"].predict(x, verbose=0)[0]
    names = im["meta"]["class_names"]
    order = prob.argsort()[::-1]
    top = [{"brand": names[i], "label": config.display(names[i]), "prob": round(float(prob[i]) * 100, 1)}
           for i in order[:3]]
    best = top[0]
    priceable = best["brand"] in (price_models() or {"meta": {"options": {}}})["meta"]["options"]
    if best["brand"] == config.OTHER:
        note = "看起來不是系統支援的品牌，請手動選擇品牌。"
    elif best["prob"] < 50:
        note = "模型不太確定，請確認品牌是否正確。"
    else:
        note = ""
    return jsonify({"top": top, "brand": best["brand"], "label": best["label"], "priceable": priceable, "note": note})


@app.route("/api/price", methods=["POST"])
def price():
    p = price_models()
    if p is None:
        return jsonify({"error": "還沒有價格模型，請先執行 python train_price.py"}), 400
    d = request.get_json()
    try:
        year, km = int(d["year"]), float(d["mileage_km"])
        X = make_input(d["brand"], d["model"], year, km, d["transmission"], d["fuel"], d["engine"])
    except (KeyError, ValueError):
        return jsonify({"error": "請把資料填完整"}), 400

    preds = predict_price(p, X)
    best = p["meta"]["best"]

    # 折舊試算：如果再開 k 年（年份當作舊 k 年、里程照目前的平均每年里程增加）
    y_max = p["meta"]["year_range"][1]
    per_year_km = km / max(1, y_max - year + 1)
    curve = []
    for k in range(0, 6):
        Xk = make_input(d["brand"], d["model"], year - k, km + k * per_year_km,
                        d["transmission"], d["fuel"], d["engine"])
        curve.append(round(predict_price(p, Xk)[best]))

    r = config.GBP_TO_TWD
    return jsonify({
        "preds": {k: {"gbp": round(v), "twd": round(v * r, -3)} for k, v in preds.items()},
        "best": best, "mape": p["meta"]["metrics"][best]["MAPE_%"],
        "curve": {"labels": ["現在"] + [f"再開 {k} 年" for k in range(1, 6)],
                  "gbp": curve, "twd": [round(c * r, -3) for c in curve]},
    })


if __name__ == "__main__":
    if price_models() is None and image_model() is None:
        raise SystemExit("找不到模型，請先執行 python train_price.py 和 python train_image.py")
    app.run(debug=True)


In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>拍照估車價｜二手車 AI 估價系統</title>
<script src="{{ url_for('static', filename='chart.umd.min.js') }}"></script>
<style>
  body { font-family: "Microsoft JhengHei", "PingFang TC", sans-serif; background:#f3f5f8; margin:0; color:#222; }
  header { background:#1f3b57; color:#fff; padding:18px 32px; }
  header h1 { margin:0; font-size:24px; }
  header p { margin:4px 0 0; opacity:.85; font-size:14px; }
  main { max-width:1100px; margin:24px auto; padding:0 16px; }
  .card { background:#fff; border-radius:12px; padding:22px; margin-bottom:18px; box-shadow:0 2px 8px rgba(0,0,0,.06); }
  .step { display:inline-block; background:#1f3b57; color:#fff; border-radius:50%; width:28px; height:28px;
          text-align:center; line-height:28px; margin-right:8px; font-weight:bold; }
  h2 { margin:0 0 14px; font-size:20px; }
  .row { display:flex; gap:20px; flex-wrap:wrap; align-items:flex-start; }
  .grid { display:grid; grid-template-columns:repeat(auto-fit, minmax(170px, 1fr)); gap:14px; }
  label { display:block; font-size:13px; color:#666; margin-bottom:4px; }
  select, input, button { font-size:16px; padding:8px 12px; border-radius:8px; border:1px solid #ccc;
                          font-family:inherit; width:100%; box-sizing:border-box; }
  button { background:#1f3b57; color:#fff; border:none; cursor:pointer; width:auto; padding:10px 26px; }
  button:hover { background:#152a3f; }
  .hint { font-size:13px; color:#888; }
  .err { color:#c00; }
  #preview { max-width:360px; max-height:240px; border-radius:10px; display:none; }
  .brand-big { font-size:34px; font-weight:bold; margin:4px 0; }
  .bar { height:22px; background:#e8edf3; border-radius:6px; margin:4px 0 10px; overflow:hidden; }
  .bar > div { height:100%; background:#3c78b4; color:#fff; font-size:13px; line-height:22px; padding-left:8px; white-space:nowrap; }
  .price-cards { display:grid; grid-template-columns:repeat(auto-fit, minmax(220px, 1fr)); gap:14px; }
  .pc { border:1px solid #e3e7ec; border-radius:10px; padding:14px; text-align:center; }
  .pc.best { border:2px solid #2e8b57; background:#f1faf4; }
  .pc .twd { font-size:28px; font-weight:bold; margin:6px 0; }
  .note { background:#fff8e1; border-left:4px solid #f0b400; padding:8px 12px; border-radius:6px; font-size:14px; }
  table { width:100%; border-collapse:collapse; font-size:14px; }
  td, th { padding:7px; border-bottom:1px solid #eee; text-align:center; }
</style>
</head>
<body>
<header>
  <h1>🚗 拍照估車價：二手車 AI 估價系統</h1>
  <p>圖片辨識（Keras CNN 遷移學習）＋ 價格預測（Scikit-Learn ／ Keras 神經網路）</p>
</header>
<main>
  <!-- 步驟 1：拍照 -->
  <div class="card">
    <h2><span class="step">1</span>拍一張車子的照片</h2>
    <div class="row">
      <div style="flex:1; min-width:260px">
        <label>上傳照片（手機可直接拍照）</label>
        <input type="file" id="photo" accept="image/*">
        <p class="hint">拍攝小技巧：拍整台車、車頭或斜前方最好認（看得到水箱罩和車標）。</p>
        <button onclick="recognize()">辨識品牌</button>
        <p id="imgErr" class="err"></p>
      </div>
      <img id="preview">
      <div id="brandResult" style="flex:1; min-width:240px; display:none">
        <div class="hint">辨識結果</div>
        <div class="brand-big" id="brandName"></div>
        <div id="topList"></div>
        <div id="brandNote" class="note" style="display:none"></div>
      </div>
    </div>
  </div>

  <!-- 步驟 2：車況 -->
  <div class="card">
    <h2><span class="step">2</span>確認品牌、輸入車況</h2>
    <div class="grid">
      <div><label>品牌（可手動修改）</label><select id="brand" onchange="fillBrand()"></select></div>
      <div><label>車型</label><select id="model"></select></div>
      <div><label>出廠年份</label><input type="number" id="year"></div>
      <div><label>里程（公里）</label><input type="number" id="km" value="60000" step="1000"></div>
      <div><label>變速箱</label><select id="trans"></select></div>
      <div><label>燃料</label><select id="fuel"></select></div>
      <div><label>排氣量（公升）</label><input type="number" id="engine" step="0.1"></div>
    </div>
    <p style="margin-top:16px"><button onclick="estimate()">開始估價</button></p>
    <p id="priceErr" class="err"></p>
  </div>

  <!-- 步驟 3：結果 -->
  <div class="card" id="priceCard" style="display:none">
    <h2><span class="step">3</span>估價結果</h2>
    <div class="price-cards" id="priceCards"></div>
    <p class="hint" id="priceNote"></p>
    <canvas id="curve" height="90" style="margin-top:12px"></canvas>
  </div>

  <!-- 模型表現 -->
  <div class="card">
    <h2>📊 模型表現（測試集）</h2>
    <div class="row">
      <div style="flex:1; min-width:300px"><b>價格預測</b><table id="pmTable"></table></div>
      <div style="flex:1; min-width:300px"><b>品牌辨識</b><table id="imTable"></table></div>
    </div>
    <p class="hint">⚠ 價格資料來自英國二手車市場（約 2020 年），台幣金額依匯率換算，僅供參考，和台灣實際行情會有差異。</p>
  </div>
</main>

<script>
let OPT = {}, RATE = 44, chart;
const fmt = n => n.toLocaleString("zh-TW");

async function init() {
  const d = await (await fetch("/api/options")).json();
  OPT = d.options; RATE = d.gbp_to_twd;
  const b = document.getElementById("brand");
  b.innerHTML = Object.keys(OPT).map(x => `<option>${x}</option>`).join("");
  const y = document.getElementById("year");
  y.min = d.year_range[0]; y.max = d.year_range[1]; y.value = d.year_range[1] - 4;
  fillBrand();
  document.getElementById("pmTable").innerHTML = "<tr><th>模型</th><th>平均誤差</th><th>誤差率</th></tr>" +
    Object.entries(d.price_metrics).map(([k, v]) =>
      `<tr${k === d.best_price_model ? ' style="background:#f1faf4;font-weight:bold"' : ""}><td>${k}</td>` +
      `<td>£${fmt(Math.round(v["MAE_£"]))}</td><td>${v["MAPE_%"]}%</td></tr>`).join("");
  document.getElementById("imTable").innerHTML = "<tr><th>模型</th><th>準確率</th><th>Macro-F1</th></tr>" +
    Object.entries(d.image_metrics).map(([k, v]) =>
      `<tr><td>${k}</td><td>${(v.accuracy * 100).toFixed(1)}%</td><td>${v.macro_f1.toFixed(3)}</td></tr>`).join("");
}

function fillBrand() {
  const o = OPT[document.getElementById("brand").value]; if (!o) return;
  document.getElementById("model").innerHTML = o.models.map(x => `<option>${x}</option>`).join("");
  document.getElementById("trans").innerHTML = o.transmission.map(x => `<option>${x}</option>`).join("");
  document.getElementById("fuel").innerHTML = o.fuel.map(x => `<option>${x}</option>`).join("");
  document.getElementById("engine").value = o.engine_median;
}

async function recognize() {
  const file = document.getElementById("photo").files[0];
  const err = document.getElementById("imgErr"); err.textContent = "";
  if (!file) { err.textContent = "請先選擇照片"; return; }
  const pv = document.getElementById("preview"); pv.src = URL.createObjectURL(file); pv.style.display = "block";
  const form = new FormData(); form.append("photo", file);
  const res = await fetch("/api/recognize", { method: "POST", body: form });
  const d = await res.json();
  if (!res.ok) { err.textContent = d.error; return; }
  document.getElementById("brandResult").style.display = "block";
  document.getElementById("brandName").textContent = d.label;
  document.getElementById("topList").innerHTML = d.top.map(t =>
    `<div style="font-size:14px">${t.label}</div><div class="bar"><div style="width:${Math.max(t.prob, 8)}%">${t.prob}%</div></div>`).join("");
  const note = document.getElementById("brandNote");
  note.style.display = d.note ? "block" : "none"; note.textContent = d.note;
  if (d.priceable) { document.getElementById("brand").value = d.brand; fillBrand(); }
}

async function estimate() {
  const body = {
    brand: v("brand"), model: v("model"), year: v("year"), mileage_km: v("km"),
    transmission: v("trans"), fuel: v("fuel"), engine: v("engine"),
  };
  const res = await fetch("/api/price", { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(body) });
  const d = await res.json();
  const err = document.getElementById("priceErr");
  if (!res.ok) { err.textContent = d.error; return; } err.textContent = "";
  document.getElementById("priceCard").style.display = "block";
  document.getElementById("priceCards").innerHTML = Object.entries(d.preds).map(([k, p]) =>
    `<div class="pc ${k === d.best ? "best" : ""}"><div>${k}${k === d.best ? "（最準）" : ""}</div>` +
    `<div class="twd">NT$ ${fmt(p.twd)}</div><div class="hint">£${fmt(p.gbp)}</div></div>`).join("");
  document.getElementById("priceNote").textContent =
    `以最準的 ${d.best} 為準，測試集平均誤差率約 ${d.mape}%。匯率以 1 英鎊 ≈ ${RATE} 台幣換算。`;
  if (chart) chart.destroy();
  chart = new Chart(document.getElementById("curve"), {
    type: "line",
    data: { labels: d.curve.labels, datasets: [{ label: "預估價格 (NT$)", data: d.curve.twd,
            borderColor: "#2e8b57", backgroundColor: "rgba(46,139,87,.12)", fill: true, tension: .3 }] },
    options: { plugins: { title: { display: true, text: "折舊試算：照目前的開車習慣，再開幾年後大概值多少？" } },
               scales: { y: { ticks: { callback: x => "NT$" + fmt(x) } } } }
  });
}
const v = id => document.getElementById(id).value;
init();
</script>
</body>
</html>


### 0-4 安裝套件與中文字型

In [ ]:
!pip install -q -U kaggle datasets
!apt-get -qq install -y fonts-noto-cjk > /dev/null
!wget -q https://cdn.jsdelivr.net/npm/chart.js@4/dist/chart.umd.min.js -O static/chart.umd.min.js
!ls
print("✅ 環境準備完成")

---
## 步驟 1：數值部分 — 二手車價格預測
### 1-1 設定 Kaggle Token
執行後會出現輸入框，**貼上你的 Token 後按 Enter**（輸入時看不到字是正常的）。

> 💡 如果有用 Colab 左側 🔑「密鑰」功能存了名為 `KAGGLE_API_TOKEN` 的密鑰，會自動讀取，不會跳出輸入框。

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("✅ 已從密鑰讀取 Token")
except Exception:
    from getpass import getpass
    os.environ["KAGGLE_API_TOKEN"] = getpass("貼上 Kaggle Token 後按 Enter：").strip()
    print("✅ Token 已設定")

### 1-2 下載價格資料
應該會看到 `audi.csv`、`bmw.csv`、`ford.csv`、`hyundi.csv`… 等檔案。出現 **401 / 403** 代表 Token 錯誤，回到 1-1 重新輸入。

In [ ]:
!kaggle datasets download -d adityadesai13/used-car-dataset-ford-and-mercedes -p data/used_cars --unzip -q
!ls data/used_cars

### 1-3 訓練價格模型（約 5～10 分鐘）
⏳ **Random Forest 那一步會停比較久**：Scikit-Learn 用不到 GPU，Colab 只有 2 個 CPU 核心，左邊 ▶ 還在轉就是還在跑。

**你應該會看到：**
```
讀取價格資料：99187 筆 → 清理後 97434 筆，9 個品牌
訓練 70395｜驗證 12423｜測試 14616
...
========== 測試集結果 ==========
  基準線(查平均價)   平均誤差 £x,xxx   誤差率 xx.x%   R² 0.xxx
  LinearRegression  ...
  RandomForest      ...
  Keras神經網路      ...
```
**怎麼判斷：** 三個機器學習模型的誤差率應該要**低於基準線**；R² 越接近 1 越好。

In [ ]:
!python train_price.py

### 1-4 看價格模型的圖表（可以直接放簡報）

In [ ]:
from IPython.display import Image, display
import os
for f in ["1_model_compare", "2_pred_vs_actual", "3_feature_importance", "4_nn_training"]:
    path = f"models/price/{f}.png"
    if os.path.exists(path):
        print(f"📊 {f}.png"); display(Image(path, width=600))
    else:
        print(f"⚠ 找不到 {path}，請確認上一格有跑到「✅ 完成」")

---
## 步驟 2：圖片部分 — 品牌辨識
### 2-1 下載車子照片並依品牌分資料夾（約 5～10 分鐘，不需要 Token）
**你應該會看到**每個品牌的照片張數，`train` 和 `test` 兩組。**每個品牌都要有照片（不能是 0）**。

In [ ]:
!python prepare_car_images.py

### 2-2 快速測試（約 3 分鐘）
只跑 1 個 epoch，**確認程式沒錯誤**。準確率很低是正常的。看到 `✅ 完成！` 就可以進下一步。

In [ ]:
!python train_image.py --quick

### 2-3 正式訓練（約 5～15 分鐘）
**你應該會看到：**
- `Epoch 1/12 ... val_accuracy: 0.xx`，**`val_accuracy` 要逐漸上升**
- 第二階段結束後會顯示「✔ 微調有效」或「✘ 微調沒有變好，保留第一階段」，兩種都正常
- 最後的「測試集結果」，以及每個品牌的 precision / recall

**怎麼判斷：** 8 類隨便猜約 12.5%，準確率要**明顯高於這個數字**。混淆矩陣可以看出哪些品牌最容易搞混。

In [ ]:
!python train_image.py

### 2-4 看品牌辨識的圖表

In [ ]:
from IPython.display import Image, display
import os
for f in ["1_confusion_matrix", "2_cnn_training", "3_examples"]:
    path = f"models/image/{f}.png"
    if os.path.exists(path):
        print(f"📊 {f}.png"); display(Image(path, width=750))
    else:
        print(f"⚠ 找不到 {path}，請確認上一格有跑到「✅ 完成」")

---
## 步驟 3：用自己拍的照片測試 ⭐ 很重要
Stanford Cars 大多是 **2012 年以前的美國車款**，台灣路上的新車款可能認不出來。

**要拍的品牌：** Toyota、Ford、Hyundai、BMW、Mercedes-Benz、Volkswagen、Audi 各 1～2 台，
再加 2～3 台**其他品牌**（Honda、Nissan、Mitsubishi…），測試模型會不會正確回答「其他品牌」。

**拍攝技巧：** 車頭或斜前方 45 度、看得到車標、拍到整台車、白天不逆光。

執行下面這格，**一次選多張上傳**。

In [ ]:
import sys, os
os.chdir("/content/used-car-price"); sys.path.insert(0, "/content/used-car-price")
for m in ["car_config", "price_utils", "plot_font", "app"]:   # 清掉之前載入過的版本
    sys.modules.pop(m, None)

import io, json, numpy as np, tensorflow as tf, keras, matplotlib.pyplot as plt
from PIL import Image as PILImage, ImageOps
from google.colab import files
from plot_font import setup_chinese_font; setup_chinese_font()
import car_config

cnn = keras.models.load_model("models/image/car_brand_cnn.keras")
names = json.load(open("models/image/metrics.json", encoding="utf-8"))["class_names"]
uploaded = files.upload()

n = len(uploaded); cols = min(n, 4); rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.2 * rows), squeeze=False)
for ax in axes.ravel(): ax.axis("off")
for ax, (fname, data) in zip(axes.ravel(), uploaded.items()):
    img = ImageOps.exif_transpose(PILImage.open(io.BytesIO(data))).convert("RGB")
    x = tf.image.resize(np.asarray(img, dtype="float32"), (224, 224))[None]
    p = cnn.predict(x, verbose=0)[0]; top = p.argsort()[::-1][:3]
    ax.imshow(img)
    ax.set_title(f"{fname[:18]}\n" + "\n".join(f"{car_config.display(names[i])} {p[i]:.0%}" for i in top), fontsize=10,
                 color="green" if p[top[0]] >= 0.5 else "darkorange")
plt.tight_layout(); plt.savefig("my_photos_result.png", dpi=110); plt.show()
print("綠色 = 信心 ≥ 50%；橘色 = 模型不太確定。結果也存成 my_photos_result.png")

---
## 步驟 4：在 Colab 裡直接打開網站 🌐
執行後下方會出現**網站畫面**，上面也有連結可以開在新分頁，可以直接上傳照片、估價。

> 如果畫面沒出現，等 10 秒再重新執行一次這格。還是不行的話：「執行階段」→「重新啟動工作階段」，再**只執行這一格**（模型檔案不會消失）。

In [ ]:
import sys, os
os.chdir("/content/used-car-price"); sys.path.insert(0, "/content/used-car-price")
for m in ["car_config", "price_utils", "plot_font", "app"]:   # 清掉之前載入過的版本
    sys.modules.pop(m, None)

import threading, time
from app import app as flask_app
if not any(t.name == "flask" for t in threading.enumerate()):
    threading.Thread(target=lambda: flask_app.run(port=5000, debug=False, use_reloader=False),
                     name="flask", daemon=True).start()
    time.sleep(3)
from google.colab import output
output.serve_kernel_port_as_window(5000, anchor_text="👉 點這裡在新分頁開啟網站")
output.serve_kernel_port_as_iframe(5000, height=1500)

---
## 步驟 5：保存模型
Colab 關掉後檔案會消失。選一種方式保存：

### 5-1 下載到電腦（在自己電腦跑網站、Demo 用）
下載後解壓縮到電腦上的專案資料夾，結構是 `used-car-price/models/price/...`、`used-car-price/models/image/...`

In [ ]:
%cd /content/used-car-price
!zip -rq car_models.zip models
from google.colab import files
files.download("car_models.zip")

### 5-2 存到 Google 雲端硬碟（下次打開 Colab 不用重新訓練）

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/used-car-price
!cp -r /content/used-car-price/models /content/drive/MyDrive/used-car-price/
print("✅ 已存到 雲端硬碟/used-car-price/models")

### 5-3 下次打開：從雲端硬碟載回模型
執行完「步驟 0」後執行這格，就可以直接跳到步驟 3、4。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/drive/MyDrive/used-car-price/models /content/used-car-price/
!ls /content/used-car-price/models/price /content/used-car-price/models/image